# BLAES Units Rhythmicity Analyses

This notebook contains code for ...

---

> *Contact: Justin Campbell (justin.campbell@hsc.utah.edu)*  
> *Version: 07/17/2024*

## 1. Import Libraries

In [1]:
import os
import re
import mne
import sys
import glob
import datetime
import numpy as np
import numba as nb
import pandas as pd
from scipy.stats import wilcoxon, ranksums, norm
from scipy.signal import butter, hilbert, filtfilt
import mat73
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.gridspec as gridspec
import seaborn as sns

# Import Blackrock Python Utilities
sys.path.append('/Users/justincampbell/Library/CloudStorage/GoogleDrive-u0815766@gcloud.utah.edu/My Drive/Research Projects/BLAESUnits/Code/Blacrock-Python-Utilities')
import brpylib

%matplotlib inline
# %config InlineBackend.figure_format='retina'
%config InlineBackend.figure_format='svg'

## 2. Set Paths & Parameters

In [2]:
# Params
fs = 30000
export = True

# Define paths
proj_path = '/Volumes/Hippocampus/BLAESUnits' # G-DRIVE Armor ATD External Drive
data_path = os.path.join(proj_path, 'Data_30k')
results_path = '/Users/justincampbell/Library/CloudStorage/GoogleDrive-u0815766@gcloud.utah.edu/My Drive/Research Projects/BLAESUnits/Results/'

# Get list of sessions
sessions = os.listdir(results_path)
sessions = [pID for pID in sessions if re.search(r'\d+$', pID)]

## 3. Load Data

In [18]:
# Load sample session
session = 'UIC20230701'
fType = session[0:3]
save_path = os.path.join(results_path, session)

# Load LFP data
if fType == 'UIC':
    data, header = loadNSX(os.path.join(data_path, session, session + '.ns6'))
    chan_idxs = [header[i]['ElectrodeID'] for i in range(len(header))]
    chan_labels = [header[i]['ElectrodeLabel'] for i in range(len(header))]
    micro_labels = [x for x in chan_labels if x.startswith('m')]
    micro_idxs = [chan_labels.index(x) for x in micro_labels]

# Load events
events = pd.read_csv(os.path.join(results_path, session, 'Events.csv'), index_col = 0)
events['Chan-Unit'] = events['Channel'].astype(str) + '-' + events['Unit'].astype(str)

# Load stim epochs
stim_epochs = pd.read_csv(os.path.join(results_path, session, 'StimEpochs.csv'), index_col = 0)

# Load waveforms
wfs = pd.read_csv(os.path.join(results_path, session, 'Waveforms.csv'), index_col = 0)

# Load stats DF
stats_df = pd.read_csv(os.path.join(results_path, 'Group', 'SpikeStats.csv'), index_col = 0)
stats_df = stats_df[(stats_df['Condition'] == 'Stim') & (stats_df['Valid'] == True)].reset_index()


UIC20230701.ns6 opened

Loading .nsx data...

UIC20230701.ns6 closed
